In [ ]:
import requests
import json
import time
import sys
import os
from bs4 import BeautifulSoup
import re

In [ ]:
# --- Configuration ---
# WARNING: Your API key is hardcoded below. For better security,
# it is strongly recommended to use an an environment variable in production.
API_KEY = os.environ.get("GEMINI_API_KEY", "")

API_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-preview-05-20:generateContent"
MAX_RETRIES = 5
UIU_URL = "https://www.uiu.ac.bd/"
LOCAL_DATA_FILE = "uiu_data.txt"

In [ ]:
# --- Helper Functions ---

def exponential_backoff_request(url, payload, retries=0):
    """
    Makes a POST request to the Gemini API with exponential backoff.
    """
    headers = {'Content-Type': 'application/json'}
    full_url = f"{url}?key={API_KEY}"

    try:
        response = requests.post(full_url, headers=headers, data=json.dumps(payload))

        if response.status_code in [429] or response.status_code >= 500:
            if retries < MAX_RETRIES:
                delay = (2 ** retries) + (time.monotonic() % 1)
                print(f"[{time.strftime('%H:%M:%S')}] Rate limited or server error ({response.status_code}). Retrying in {delay:.2f} seconds...", file=sys.stderr)
                time.sleep(delay)
                return exponential_backoff_request(url, payload, retries + 1)
            else:
                # FIX: Ensure it always raises the exception if retries fail
                response.raise_for_status()

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        print(f"\n[ERROR] API Request failed: {e}", file=sys.stderr)
        return None

In [ ]:
def scrape_uiu_website(url):
    """
    Scrapes the main text content from the UIU website.
    """
    print(f"Scraping content from {url}...")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # Focus on main readable content (paragraphs, headings)
        texts = []
        for tag in soup.find_all(['p', 'h1', 'h2', 'h3', 'li']):
            # Clean up text: remove multiple spaces and strip
            text = ' '.join(tag.get_text(strip=True).split())
            if len(text) > 30: # Ignore very short strings
                texts.append(text)

        content = "\n\n---\n\n".join(texts)

        # Save to local file
        with open(LOCAL_DATA_FILE, 'w', encoding='utf-8') as f:
            f.write(content)

        print(f"Successfully scraped content and saved to {LOCAL_DATA_FILE}.")
        return content

    except requests.exceptions.RequestException as e:
        print(f"\n[ERROR] Failed to scrape website ({url}): {e}", file=sys.stderr)
        return None

In [ ]:
def load_or_scrape_data():
    """
    Loads local data or scrapes the website if the file doesn't exist.
    """
    if os.path.exists(LOCAL_DATA_FILE):
        # We only print the loading message if the file is loaded, not scraped
        # The scraping function prints its own success message.
        with open(LOCAL_DATA_FILE, 'r', encoding='utf-8') as f:
            data = f.read()
        return data
    else:
        print(f"Local data file not found. Initializing by scraping {UIU_URL}.")
        return scrape_uiu_website(UIU_URL)

In [ ]:
def search_local_data(query, local_text):
    """
    Performs a robust keyword search on the local text, prioritizing newly learned (API) data.
    Returns the surrounding context if a match is found.
    """
    if not local_text:
        return None

    # Use simple word tokenization for matching
    keywords = re.findall(r'\b\w+\b', query.lower())

    # Split text into paragraphs/sections based on the separator used during scraping
    sections = local_text.split("\n\n---\n\n")

    # 1. Prioritize search in newly saved (API) sections
    relevant_api_lines = []

    for section in sections:
        section_lower = section.lower()
        if "" in section_lower:
            # Check the actual factual content part, which starts after the header
            factual_content = section_lower.split("factual content:", 1)[-1]

            # For newly saved comprehensive answers, only one meaningful keyword is needed for a match
            match_count = sum(1 for keyword in keywords if len(keyword) > 3 and keyword in factual_content)

            if match_count >= 1:
                relevant_api_lines.append(section)
                if len(relevant_api_lines) >= 1: # Only need the best single match from API saves
                    return "\n\n".join(relevant_api_lines)

    # 2. Fallback to general scraped content search (original logic for static scrape)
    relevant_scraped_lines = []
    for section in sections:
        section_lower = section.lower()
        if not "" in section_lower: # Don't re-check API saves
            # Count keywords longer than 3 characters (to ignore 'a', 'the', 'is', etc.)
            match_count = sum(1 for keyword in keywords if len(keyword) > 3 and keyword in section_lower)

            # If we have a decent number of matching keywords (>= 2), consider it relevant
            if match_count >= 2:
                relevant_scraped_lines.append(section)
                if len(relevant_scraped_lines) >= 3: # Limit to top 3 snippets for context
                    break

    return "\n\n".join(relevant_scraped_lines) if relevant_scraped_lines else None

In [ ]:
def is_conversational_response(text):
    """Simple heuristic to avoid saving greetings or refusals to the local file."""
    text_lower = text.lower().strip()
    return any(text_lower.startswith(word) for word in ["hello", "hi", "i am designed", "thank you", "thanks", "i encountered", "sorry"])

In [ ]:
def update_local_data(user_query, bot_response_text):
    """
    Appends the bot's factual response to the local file.
    This is called when a factual answer is obtained from the API (not locally).
    """
    if bot_response_text and not is_conversational_response(bot_response_text):
        timestamp = time.strftime('%Y-%m-%d %H:%M:%S')

        # Use a clear format to store the newly learned information
        new_data = (
            f"\n\n---\n\n"
            f"[Source: API/Web Search]\n"
            f"[Query: {user_query}]\n"
            f"[Date Added: {timestamp}]\n"
            f"Factual Content:\n{bot_response_text}"
        )
        try:
            with open(LOCAL_DATA_FILE, 'a', encoding='utf-8') as f:
                f.write(new_data)
            return True
        except IOError as e:
            print(f"[ERROR] Could not write to local file: {e}", file=sys.stderr)
            return False
    return False

In [ ]:
def get_gemini_response(query, chat_history, use_grounding=False, context_text=None):
    """
    Generates a response using the Gemini API.
    Returns: (full_display_text, was_grounded_or_contextual, raw_bot_text_for_saving)
    """

    system_instruction = (
        "You are the United International University (UIU) Assistant. "
        "Your primary role is to answer questions related to UIU, its admissions, "
        "programs, faculties, campus life, and events. "
        "You can also handle friendly, non-knowledge-based greetings (e.g., 'Hello', 'Thank you'). "
        "If the user asks a factual question unrelated to UIU (e.g., 'What is the capital of France?'), "
        "you MUST politely decline, stating: 'I am designed to assist only with questions concerning United International University (UIU).' "
    )

    # 1. Prepare contents for API
    # If local context is provided, prepend it to the current user query to guide the model.
    if context_text:
        # --- MODIFIED SECTION START ---
        # Create a new query part that includes the scraped text as context
        # The instruction is changed to be more neutral if the answer is not found.
        full_query = (
            f"Based *only* on the following UIU information, answer the user's question: '{query}'. "
            f"If the information below does not contain the answer, clearly state that the answer is not available in your local data. Take user permission and search web. "
            f"\n\n--- UIU Local Data Context ---\n{context_text}"
        )
        # --- MODIFIED SECTION END ---
        current_contents = chat_history + [{"role": "user", "parts": [{"text": full_query}]}]
    else:
        current_contents = chat_history + [{"role": "user", "parts": [{"text": query}]}]

    # 2. Prepare payload
    payload = {
        "contents": current_contents,
        "systemInstruction": {"parts": [{"text": system_instruction}]}
    }

    # 3. Add tools (Google Search Grounding) if requested (for external fallback)
    if use_grounding:
        payload["tools"] = [{"google_search": {}}]

    response_data = exponential_backoff_request(API_URL, payload)

    if not response_data:
        return "Sorry, I encountered a network error while trying to reach the API.", False, None

    try:
        candidate = response_data.get('candidates', [{}])[0]
        # This is the raw text response without citations
        bot_text_clean = candidate.get('content', {}).get('parts', [{}])[0].get('text')

        if not bot_text_clean:
            return "Sorry, I received an empty or unparsable response from the Gemini API.", False, None

        # Process grounding sources for citation display
        sources_text = ""
        grounding_metadata = candidate.get('groundingMetadata')
        if grounding_metadata and grounding_metadata.get('groundingAttributions'):
            sources = grounding_metadata['groundingAttributions']
            source_list = "\n\n(Sources: " + ", ".join([
                f"[{i+1}] {s['web']['title']}"
                for i, s in enumerate(sources) if s.get('web')
            ]) + ")"
            sources_text = source_list

        # Return: 1. Text with sources, 2. Whether local context was NOT used (i.e., API did the work), 3. Raw text for saving
        was_api_generated = (context_text is None)
        return bot_text_clean + sources_text, was_api_generated, bot_text_clean

    except Exception as e:
        return f"[ERROR] Failed to process Gemini response: {e}", False, None

In [ ]:
def main_chat_loop():
    """
    Main function to run the interactive command-line chat.
    This function implements the new logic to prompt the user for permission
    before performing a web search if the local file does not contain the answer.
    """
    # ... [Configuration and Initialization code remains the same] ...
    if not API_KEY:
        print("Please set the GEMINI_API_KEY environment variable.")
        return

    local_uiu_text = load_or_scrape_data()
    chat_history = []

    print("\n" + "="*50)
    print("🤖 UIU Hybrid Chatbot (Type 'exit' or 'quit' to end)")
    print("   (Answers UIU questions using local file first, then the web.)")
    print("="*50)

    while True:
        try:
            user_input = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break

        if user_input.lower() in ['exit', 'quit']:
            break

        if not user_input.strip():
            continue

        print("🤖 Thinking...", end="", flush=True)

        use_grounding = False
        context_for_gemini = None

        simple_queries = ["hello", "hi", "how are you", "what is your name", "thank you", "thanks"]
        is_simple_query = any(keyword in user_input.lower() for keyword in simple_queries)

        if not is_simple_query:
            # 1. Attempt to search the local file first
            local_context = search_local_data(user_input, local_uiu_text)

            if local_context:
                # Answer Found Locally: Use Gemini to summarize the local context
                context_for_gemini = local_context
                print("\r[Local Data Found] Generating response...", end="", flush=True)
            else:
                # Answer Not Found Locally: Go straight to permission prompt

                # Clear "Thinking..." message
                print("\r" + " " * 50 + "\r", end="", flush=True)
                print("[No Local Data] The answer isn't in my UIU file. ", end="", flush=True)

                # --- NEW LOGIC: Ask for permission ---
                permission = input("Would you like me to search the web for the latest UIU information? (yes/no): ")

                if permission.lower().startswith('y'):
                    use_grounding = True
                    # Re-print status message for consistency
                    print("🤖 Searching the web...", end="", flush=True)
                else:
                    # User declined web search
                    print("🤖 I am sorry, I cannot answer your question without searching the web. Please try a different UIU-related query or allow the search.")
                    continue
                # -------------------------------------
        else:
            # For simple talk, just use the API without grounding or local context
            print("\r[Conversation] Generating response...", end="", flush=True)


        # Call the Gemini API
        full_bot_response, was_api_generated, bot_response_text_clean = get_gemini_response(
            query=user_input,
            chat_history=chat_history,
            use_grounding=use_grounding,
            context_text=context_for_gemini
        )

        # Clear status message (clear wide area to handle the prompt)
        print("\r" + " " * 100 + "\r", end="", flush=True)

        if full_bot_response and not full_bot_response.startswith("[ERROR]"):

            # --- MODIFIED SECTION START ---
            # Check for a strong refusal that indicates the local context failed
            # Added more keywords to reliably detect a failure response.
            needs_web_search = context_for_gemini and (
                "does not specify" in full_bot_response.lower() or
                "not contain the answer" in full_bot_response.lower() or
                "not available in your local data" in full_bot_response.lower() or
                "could not find" in full_bot_response.lower()
            )
            # --- MODIFIED SECTION END ---


            if needs_web_search:
                # The model confirmed the local data was insufficient. We must now ask the user for permission.
                print(f"ChatBot: {full_bot_response}") # Display the first part of the response (saying local data failed)

                # Prompt for permission
                permission = input("Would you like me to search the web for the latest UIU information? (yes/no): ")

                if permission.lower().startswith('y'):
                    print("🤖 Searching the web...", end="", flush=True)

                    # Rerun the API call with grounding enabled and no local context
                    full_bot_response, was_api_generated, bot_response_text_clean = get_gemini_response(
                        query=user_input,
                        chat_history=chat_history,
                        use_grounding=True, # Enable Grounding
                        context_text=None    # Remove local context
                    )
                    use_grounding = True # Ensure this flag is set for the saving logic below
                    # Clear status message again
                    print("\r" + " " * 100 + "\r", end="", flush=True)
                else:
                    # User declined the second search
                    print("🤖 Understood. I will not search the web. Please ask another question.")
                    continue # Skip the rest of the loop

            # --- Normal Response Processing resumes here ---

            final_display_response = full_bot_response

            # 1. Add local indicator if local data was used (and the response was factual, not the refusal)
            if context_for_gemini and not needs_web_search:
                final_display_response = "[Local File Source] " + final_display_response

            # 2. Update the file if the response came from the web search
            if was_api_generated and use_grounding:
                if update_local_data(user_input, bot_response_text_clean):
                    local_uiu_text = load_or_scrape_data()
                    print(f"[{time.strftime('%H:%M:%S')}] Local data updated (new knowledge saved) and reloaded.")

            print(f"ChatBot: {final_display_response}")

            # Add user/model turn to history for conversation continuity (use the clean version for history)
            chat_history.append({"role": "user", "parts": [{"text": user_input}]})
            chat_history.append({"role": "model", "parts": [{"text": bot_response_text_clean}]})
        else:
            print("ChatBot: I encountered a critical error and cannot respond. Check the error log above.", file=sys.stderr)

    print("\nChat session ended. Goodbye!")

if __name__ == "__main__":
    main_chat_loop()